# Experiment: ConvNeXt-Tiny for Audio Genre Classification

## Why ConvNeXt?

ResNet34 has been a reliable backbone, but it was designed in 2015 for natural images. ConvNeXt (Liu et al., 2022) modernised the plain convolutional network architecture by borrowing ideas from Vision Transformers without the attention overhead:

- Larger depthwise convolution kernels (7x7 instead of 3x3)
- Inverted bottleneck structure
- LayerNorm instead of BatchNorm
- Fewer activation functions per block

On ImageNet, ConvNeXt-Tiny achieves 82.1% accuracy vs ResNet50's 76.1% with comparable parameters. Since our mel spectrograms are 2D images, image classification backbones transfer directly.

Reference: Liu et al., "A ConvNet for the 2020s", CVPR 2022. https://arxiv.org/abs/2201.03545

In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from pathlib import Path
import wandb

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DATA_ROOT = Path('/teamspace/studios/this_studio/data/messy_mashup')
CHKPT_DIR = Path('/teamspace/studios/this_studio/new_chkpt')
CHKPT_DIR.mkdir(exist_ok=True)

GENRES = ['blues', 'classical', 'country', 'disco', 'hiphop',
          'jazz', 'metal', 'pop', 'reggae', 'rock']
GENRE_TO_IDX = {g: i for i, g in enumerate(GENRES)}

SR = 22050
DURATION = 5.0
N_MELS = 128
N_FFT = 1024
HOP = 512

BATCH_SIZE = 32
LR = 3e-4
EPOCHS = 15
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

## Dataset

Audio is pre-loaded into RAM during `__init__`. This avoids disk I/O bottlenecks during training where repeated file reads were causing DataLoader workers to freeze.

In [ ]:
class AudioGenreDataset(Dataset):
    def __init__(self, root, genres, genre_to_idx, sr=SR, duration=DURATION,
                 split='train', augment=False):
        self.sr = sr
        self.duration = duration
        self.n_samples = int(sr * duration)
        self.augment = augment
        self.genre_to_idx = genre_to_idx
        self.samples = []
        self.labels = []

        print(f'Pre-loading {split} audio into RAM...')
        for genre in genres:
            genre_dir = root / split / genre
            if not genre_dir.exists():
                continue
            for f in sorted(genre_dir.glob('*.wav')):
                try:
                    waveform, file_sr = torchaudio.load(str(f))
                    if file_sr != sr:
                        waveform = T.Resample(file_sr, sr)(waveform)
                    waveform = waveform.mean(dim=0)
                    self.samples.append(waveform)
                    self.labels.append(genre_to_idx[genre])
                except Exception as e:
                    print(f'Skipping {f.name}: {e}')
        print(f'Loaded {len(self.samples)} files.')

        # GPU mel transform (applied in __getitem__ after cropping)
        self.mel_transform = nn.Sequential(
            T.MelSpectrogram(sample_rate=sr, n_fft=N_FFT, hop_length=HOP,
                             n_mels=N_MELS, f_min=20, f_max=8000),
            T.AmplitudeToDB(top_db=80)
        )

    def _crop(self, waveform):
        n = waveform.shape[0]
        if n >= self.n_samples:
            if self.augment:
                start = random.randint(0, n - self.n_samples)
            else:
                start = (n - self.n_samples) // 2
            return waveform[start:start + self.n_samples]
        return torch.nn.functional.pad(waveform, (0, self.n_samples - n))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        waveform = self._crop(self.samples[idx])
        mel = self.mel_transform(waveform.unsqueeze(0))  # (1, n_mels, T)
        mel = mel.repeat(3, 1, 1)                        # (3, n_mels, T) for RGB backbone
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)   # per-sample normalisation
        return mel, self.labels[idx]

## Model: ConvNeXt-Tiny

We load ImageNet pretrained weights and replace the classifier head. The final layer outputs 10 logits for 10 genres.

In [ ]:
class AudioConvNeXtTiny(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()
        self.backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        # Replace the classifier
        in_features = self.backbone.classifier[2].in_features
        self.backbone.classifier[2] = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(in_features, num_classes)
        )

    def forward(self, x):
        return self.backbone(x)


model = AudioConvNeXtTiny(num_classes=10, dropout=0.3)
model = nn.DataParallel(model)
model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

In [ ]:
train_dataset = AudioGenreDataset(DATA_ROOT, GENRES, GENRE_TO_IDX, split='train', augment=True)
val_dataset   = AudioGenreDataset(DATA_ROOT, GENRES, GENRE_TO_IDX, split='val',   augment=False)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## Training Setup

- **Optimiser:** AdamW with weight decay 0.01. AdamW separates L2 regularisation from the gradient update, which works better with LayerNorm layers in ConvNeXt.
- **Scheduler:** CosineAnnealingLR. Smoothly reduces the learning rate from LR to near zero over training, avoiding abrupt drops.
- **AMP:** Mixed precision (FP16 compute, FP32 master weights) for roughly 2x memory efficiency.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = GradScaler()

wandb.init(
    project='22f3003226-t12026',
    entity='dlgeneai-team',
    name='convnext-tiny-audio',
    config={
        'model': 'convnext_tiny',
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'n_mels': N_MELS,
        'duration': DURATION,
        'dropout': 0.3,
        'label_smoothing': 0.1,
        'weight_decay': 0.01,
    }
)

In [ ]:
def run_epoch(loader, model, criterion, optimizer, scaler, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch_idx, (mel, labels) in enumerate(loader):
        mel = mel.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with autocast():
            logits = model(mel)
            loss = criterion(logits, labels)

        if train:
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(dim=1) == labels).sum().item()
        total += labels.size(0)

        if train and batch_idx % 20 == 0:
            print(f'  Batch {batch_idx}/{len(loader)} | Loss: {loss.item():.4f}')

    return total_loss / total, correct / total

In [ ]:
best_val_acc = 0.0
save_path = CHKPT_DIR / 'ConvNeXtTiny_best.pth'

for epoch in range(1, EPOCHS + 1):
    print(f'\nEpoch {epoch}/{EPOCHS}')

    train_loss, train_acc = run_epoch(train_loader, model, criterion, optimizer, scaler, train=True)
    scheduler.step()

    with torch.no_grad():
        val_loss, val_acc = run_epoch(val_loader, model, criterion, optimizer, scaler, train=False)

    print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
    print(f'  Val   Loss: {val_loss:.4f} | Val   Acc: {val_acc:.4f}')

    wandb.log({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_acc': train_acc,
        'val_loss': val_loss,
        'val_acc': val_acc,
        'lr': scheduler.get_last_lr()[0]
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), save_path)
        print(f'  Saved new best checkpoint (val_acc={val_acc:.4f})')

wandb.finish()
print(f'\nTraining complete. Best val accuracy: {best_val_acc:.4f}')